In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as tk

In [ ]:
params = {
   'axes.labelsize': 21,
   'font.size': 16,
   'font.family': 'sans-serif',
   'font.serif': 'Arial',
   'legend.fontsize': 18,
   'xtick.labelsize': 18,
   'ytick.labelsize': 18,
   'axes.labelpad': 15,

   'figure.figsize': [10,8], # value in inches based on dpi of monitor
   'figure.dpi': 105.5, # My monitor has a dpi of around 105.5 px/inch

   'axes.grid': True,
   'grid.linestyle': '-',
   'grid.alpha': 0.25,
   'axes.linewidth': 1,
   'figure.constrained_layout.use': True,


   # Using Paul Tol's notes:
   'axes.prop_cycle':
      mpl.cycler(color=['#4477aa', # blue
                        '#ee6677', # red/pink
                        '#228833', # green
                        '#aa3377', # purple
                        '#66ccee', # cyan
                        '#ccbb44', # yellow
                        '#bbbbbb', # grey
                        ]),

      # Pick either the cycler above, or the cycler below:

      # (mpl.cycler(color=['#4477aa', # blue
      #                     '#ee6677', # red/pink
      #                     '#228833', # green
      #                     '#aa3377', # purple
      #                     '#66ccee', # cyan
      #                     '#ccbb44', # yellow
      #                     '#bbbbbb', # grey
      #                     ]) +
      #   mpl.cycler(linestyle=['-', # solid
      #                         '--', # dashed
      #                         ':', # dotted
      #                         '-.', # dash dot
      #                         (0, (3, 1, 1, 1, 1, 1)), # narrow dash dot dot
      #                         (0, (1, 2, 5, 2, 5, 2)), # dash dash dot
      #                         (0, (5, 2.5, 1, 2.5, 1, 2.5)), # dash dot dot
      #                         ])),

   'lines.linewidth': 2.5,

   'image.cmap': 'jet',
}


plt.rcParams.update(params)

## MIGDAL data

In [ ]:
wA = 26.4
wC = 34.3
def charge_to_gain(charge, pA, pC):
    charge = np.array(charge)
    wval = 1/((1/wA - 1/wC)*(pA/(pA+3.01*pC)) + 1/wC)
    num_elec = charge*1E-12/1.6E-19
    prim_elec = 5.2E3/wval
    return (num_elec/prim_elec)

In [ ]:
#Pure CF4 data
voltages_60Torr_cf4 = [610, 620, 600, 595, 580, 615, 590, 605]
pressures_60Torr_cf4 = np.ones(len(voltages_60Torr_cf4))*60
argon_per_60Torr_cf4 = np.zeros(len(voltages_60Torr_cf4))

ito_charge_60Torr_cf4 = [8.77, 13.54, 5.487, 4.260, 2.046, 10.921, 3.35, 6.939]
ito_charge_errs_60Torr_cf4 = [0.018, 0.034, 0.02, 0.019, 0.01, 0.025, 0.019, 0.015]
ito_res_60Torr_cf4 = [16.8, 16.2, 21.4, 25.5, 45.2, 16.5, 30, 18.3]
ito_res_errs_60Torr_cf4 = [0.2, 0.3, 0.4, 0.5, 0.5, 0.2, 0.6,  0.2]

#30% Ar data
voltages_60Torr_3070 = [540,550,555,560,565,570, 575, 580,585,590]
pressures_60Torr_3070 = np.ones(len(voltages_60Torr_3070))*60
argon_per_60Torr_3070 = np.ones(len(voltages_60Torr_3070))*30

ito_charge_60Torr_3070 = [1.364,2.173,2.795, 3.602,4.694,6.114,7.865, 10.035,12.737,16.079]
ito_charge_errs_60Torr_3070 = [0.009, 0.018, 0.023, 0.019, 0.007, 0.024, 0.018, 0.021, 0.024, 0.008]
ito_res_60Torr_3070 = [35.3, 15.2, 19.7, 17.2, 64.1, 24.5, 29.5, 15.8, 15, 42.7]
ito_res_errs_60Torr_3070 = [0.4, 0.2, 0.4, 0.3, 0.5, 0.5, 0.5, 0.2, 0.2, 0.4]

## Simulation data

In [ ]:
voltages_sim_purecf4 = [590,600,610,615,620]
voltages_sim_3070 = [550,560,570,580,590]
gains_sim_purecf4 = [0,0,0,0,0]
err_sim_purecf4 = [0,0,0,0,0]
gains_sim_3070 = [0,0,0,0,0]
err_sim_3070 = [0,0,0,0,0]

## Pure CF4

In [ ]:
index = 0
for voltage in voltages_sim_purecf4:
  #importing data
  gain_list = []
  for i in range(1,5001,1):
    try:
      infile = open("/Users/tomszwarcer/Documents/MIGDAL/UPDATE/gasgain_output/output"+str(voltage)+"_0/" + str(i) + ".csv","r")
    except FileNotFoundError:
      continue
    all = infile.readlines()
    gain = int(all[0])
    gain_list.append(gain)

  n_bins = 35

  #calculate mean
  total_gain = 0
  num_runs = len(gain_list)
  for gain in gain_list:
    total_gain += gain
  mean = total_gain/num_runs

  #calculate variance
  var = 0
  for gain in gain_list:
    var += (gain - mean)**2
  var = var/num_runs
  se = np.sqrt(var)/np.sqrt(num_runs)

  gains_sim_purecf4[index] = mean
  err_sim_purecf4[index] = se

  #Plot histogram
  entries, bc, p = plt.hist(gain_list, bins=n_bins, density=False)
  plt.xlabel("Gain")
  plt.ylabel("Counts")
  plt.title("Gain distribution [Pure CF4, 60 Torr, dV = "+str(voltage)+"]")
  plt.text(x=0.5*plt.xlim()[1],y=0.75*plt.ylim()[1],s="mean = " + str(int(round(mean,0))) + "\nSD = " + str(int(round(se,0))))
  plt.savefig("plots/"+str(voltage)+"_0.png")
  plt.close()
  index += 1

## 30% Ar

In [ ]:
index = 0
for voltage in voltages_sim_3070:
  #importing data
  gain_list = []
  for i in range(1,5001,1):
    try:
      infile = open("/Users/tomszwarcer/Documents/MIGDAL/UPDATE/gasgain_output/output"+str(voltage)+"_30/" + str(i) + ".csv","r")
    except FileNotFoundError:
      continue
    all = infile.readlines()
    gain = int(all[0])
    gain_list.append(gain)

  n_bins = 35

  #calculate mean
  total_gain = 0
  num_runs = len(gain_list)
  for gain in gain_list:
    total_gain += gain
  mean = total_gain/num_runs

  #calculate variance
  var = 0
  for gain in gain_list:
    var += (gain - mean)**2
  var = var/num_runs
  se = np.sqrt(var)/np.sqrt(num_runs)

  gains_sim_3070[index] = mean
  err_sim_3070[index] = se

  #Plot histogram
  entries, bc, p = plt.hist(gain_list, bins=n_bins, density=False)
  plt.xlabel("Gain")
  plt.ylabel("Counts")
  plt.title("Gain distribution [CF4:Ar 70:30, 60 Torr, dV = "+str(voltage)+"]")
  plt.text(x=0.5*plt.xlim()[1],y=0.75*plt.ylim()[1],s="mean = " + str(int(round(mean,0))) + "\nSD = " + str(int(round(se,0))))
  plt.savefig("plots/"+str(voltage)+"_30.png")
  plt.close()
  index += 1

## Comparison

In [ ]:


print(voltages_sim_purecf4)
print(gains_sim_purecf4)
print(err_sim_purecf4)
print("\n")
print(voltages_sim_3070)
print(gains_sim_3070)
print(err_sim_3070)

In [ ]:
ax = plt.axes()

#Pure CF4
ax.scatter(voltages_60Torr_cf4,charge_to_gain(ito_charge_60Torr_cf4, 0, 60),label="MIGDAL Data (0% Ar)", s = 50.0)
ax.scatter(voltages_sim_purecf4,gains_sim_purecf4,label="Simulation (0% Ar)", marker='x', s=60.0,color='#4477aa')
ax.errorbar(voltages_sim_purecf4,gains_sim_purecf4,yerr=err_sim_purecf4,elinewidth = 1.2, ecolor = '#4477aa', ls='none')
ax.set_xlabel("GEM dV [V]")
ax.set_ylabel("Gain")

#30% Ar
ax.scatter(voltages_60Torr_3070,charge_to_gain(ito_charge_60Torr_3070, 18, 42),label="MIGDAL measurement (30% Ar)")
ax.scatter(voltages_sim_3070,gains_sim_3070,label="Simulation (30% Ar)", marker='x', s=60.0,color='#ee6677')
ax.errorbar(voltages_sim_3070,gains_sim_3070,yerr=err_sim_3070,elinewidth = 1.2, ecolor = '#ee6677', ls='none')

ax.text(613,645000,'[60 Torr]')
ax.legend(bbox_to_anchor=(0.56,1.17),loc='upper right',mode='expand',ncols=2,frameon=False,fontsize=18)
#ax.set_yscale("log")


plt.savefig("plots/comparison.png")
plt.close()